In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.cluster import KMeans
import numpy as np

batch_size = 128
latent_dim = 64
epochs = 15
lr = 0.001
num_clusters = 10
transform = transforms.Compose([
    transforms.ToTensor(),
])
dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*32*3, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 512),
            nn.ReLU(),
            nn.Linear(512, 32*32*3),
            nn.Sigmoid()
        )
    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z
model = Autoencoder()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

for epoch in range(epochs):
    total_loss = 0
    for images, _ in loader:
        images = images.view(images.size(0), -1)
        outputs, _ = model(images)
        loss = criterion(outputs, images)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader)}")
features = []
with torch.no_grad():
    for images, _ in loader:
        _, z = model(images)
        features.append(z.numpy())
features = np.concatenate(features, axis=0)
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
clusters = kmeans.fit_predict(features)
print("Кластеризация завершена")

100%|██████████| 170M/170M [00:20<00:00, 8.32MB/s]


Epoch 1, Loss: 0.03010463148660367
Epoch 2, Loss: 0.02080709674417058
Epoch 3, Loss: 0.018170722005198068
Epoch 4, Loss: 0.016483461691538238
Epoch 5, Loss: 0.015488374611491438
Epoch 6, Loss: 0.01470091644569736
Epoch 7, Loss: 0.01408181885195434
Epoch 8, Loss: 0.013612813895087108
Epoch 9, Loss: 0.013204634637402757
Epoch 10, Loss: 0.012906167227441392
Epoch 11, Loss: 0.012645516856609251
Epoch 12, Loss: 0.01242072401744554
Epoch 13, Loss: 0.01220261524467136
Epoch 14, Loss: 0.011924926896610528
Epoch 15, Loss: 0.011734129439401048
Кластеризация завершена
